# 23CSE301 Machine Learning - Capstone Project

## Classification Track - Part A

### Bank Marketing Dataset

**Algorithms:**
1. Logistic Regression
2. K-Nearest Neighbors
3. Gaussian Naive Bayes
4. Decision Tree Classifier
5. Support Vector Machine

---

### Problem Statement
The objective of this classification project is to predict whether a bank customer will subscribe to a term deposit (binary target variable `y`: `yes` or `no`) based on direct telemarketing campaign data from a Portuguese banking institution. The dataset encompasses client demographics, financial indicators, past contact interactions, and macroeconomic attributes. Accurately predicting customer subscription propensity enables the institution to optimize marketing resource allocation, target high-probability prospective clients, and improve campaign efficiency.

In accordance with the 23CSE301 Capstone Project Guidelines, this notebook implements Part A of the Classification Track, covering the five core algorithms evaluated in Review 1. All models are trained and tested on an identical stratified split with strict prevention of data leakage.


## 2. Import Required Libraries

This section imports the necessary libraries for data processing, exploratory data analysis, pipeline creation, model training, hyperparameter tuning, and performance evaluation.

A fixed random seed (`RANDOM_STATE = 42`) is established to guarantee reproducibility across all data splits and randomized estimators.


In [ ]:
# Core data manipulation and numerical libraries
import os
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn model selection and validation
from sklearn.model_selection import train_test_split, GridSearchCV

# Scikit-learn preprocessing and pipeline utilities
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Scikit-learn classification algorithms (Part A)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

# Scikit-learn evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Plot formatting and styling
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.autolayout"] = True

# Deterministic random seed
RANDOM_STATE = 42


## 3. Load Dataset

Loading the Bank Marketing dataset (`bank-full.csv`). The file uses semicolon delimiters (`sep=";"`). The path is verified to ensure robust loading whether executed from the project root or the notebooks directory.


In [ ]:
# Locate bank-full.csv (checking current directory and common relative data paths)
csv_filename = "bank-full.csv"
if os.path.exists(csv_filename):
    csv_path = csv_filename
elif os.path.exists(os.path.join("data", csv_filename)):
    csv_path = os.path.join("data", csv_filename)
elif os.path.exists(os.path.join("..", "data", csv_filename)):
    csv_path = os.path.join("..", "data", csv_filename)
else:
    csv_path = csv_filename

df = pd.read_csv(csv_path, sep=";")
print(f"Dataset successfully loaded from: {csv_path}")


In [ ]:
# Display first 5 rows
print("--- First 5 Rows ---")
display(df.head())

# Display last 5 rows
print("--- Last 5 Rows ---")
display(df.tail())

# Dataset shape
print(f"\nDataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

# Column names and data types
print("\n--- Column Names & Data Types ---")
display(df.dtypes.to_frame(name="Data Type"))

# DataFrame info
print("\n--- DataFrame Information ---")
df.info()


## 4. Dataset Audit

A rigorous dataset audit assesses data hygiene prior to model construction:
- Verification of missing values (`NaN` / `null`)
- Detection of duplicate records
- Examination of numerical feature distributions and ranges
- Examination of categorical unique value counts and cardinality
- Identification and enumeration of implicit missing values recorded as `"unknown"`


In [ ]:
# Missing values audit
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing_counts, "Percentage (%)": missing_pct})
print("--- Missing Values Audit ---")
display(missing_df)

# Duplicate records audit
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate Rows Detected: {duplicate_count}")


In [ ]:
# Statistical summary of numerical features
print("--- Numerical Features Statistical Summary ---")
display(df.describe().T)

# Statistical summary of categorical features
print("\n--- Categorical Features Statistical Summary ---")
display(df.describe(include=["object"]).T)


In [ ]:
# Categorical unique values audit
print("--- Categorical Unique Values ---")
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    print(f"Column '{col}' ({df[col].nunique()} unique): {df[col].unique().tolist()[:8]}")

# Audit 'unknown' values across columns
unknown_counts = (df == "unknown").sum()
unknown_df = pd.DataFrame({
    "Unknown Count": unknown_counts[unknown_counts > 0],
    "Percentage (%)": ((unknown_counts[unknown_counts > 0] / len(df)) * 100).round(2)
})
print("\n--- Implicit Missing Values ('unknown') Audit ---")
display(unknown_df)
